In [1]:
import random
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
import ruptures as rpt


# ------------------------
# 設定
# ------------------------
NUM_COINS = 100
DATA_DIR = Path.cwd() / "coingecko_by_coin"
OUTPUT_CSV = Path.cwd() / "change_point_pairs.csv"

AR_LAGS = 5
CHANGE_MODEL = "rbf"
CHANGE_PEN = 10

PRICE_COL_CANDIDATES = ["price", "Price", "close", "Close"]
TIME_COL_CANDIDATES = ["timestamp", "Timestamp", "date", "Date", "time", "Time"]


def log(msg: str):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")


# ------------------------
# CSV一覧を取得
# ------------------------
def find_coin_files(data_dir: Path):
    if not data_dir.exists():
        raise FileNotFoundError(f"フォルダが見つかりません: {data_dir}")

    files = sorted(data_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"CSVファイルが見つかりません: {data_dir}")

    return files


# ------------------------
# 列名を自動判定
# ------------------------
def detect_column(df: pd.DataFrame, candidates: list, col_type: str):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        f"{col_type}列が見つかりません。候補: {candidates}, 実際の列: {list(df.columns)}"
    )


# ------------------------
# CSV読み込み
# ------------------------
def load_coin_csv(file_path: Path):
    df = pd.read_csv(file_path)

    time_col = detect_column(df, TIME_COL_CANDIDATES, "時刻")
    price_col = detect_column(df, PRICE_COL_CANDIDATES, "価格")

    df = df[[time_col, price_col]].copy()
    df.columns = ["timestamp", "price"]

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df = df.dropna(subset=["timestamp", "price"]).copy()
    df = df[df["price"] > 0].copy()
    df = df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    if len(df) == 0:
        raise ValueError(f"有効なデータがありません: {file_path.name}")

    return df


# ------------------------
# ファイル名から銘柄名を推定
# ------------------------
def symbol_from_filename(file_path: Path):
    return file_path.stem


# ------------------------
# AR を価格そのものに適用
# ------------------------
def apply_ar_price_in_sample(df: pd.DataFrame, symbol: str, lags: int = 5):
    series = pd.Series(df["price"].values.astype(float))

    if len(series) <= lags + 5:
        log(f"[{symbol}] データが短すぎるため AR をスキップします")
        return None, None

    model = AutoReg(series, lags=lags, old_names=False)
    res = model.fit()

    fitted = res.fittedvalues
    actual = series[lags:]

    min_len = min(len(actual), len(fitted))
    actual = actual.iloc[:min_len]
    fitted = fitted.iloc[:min_len]

    mse = ((actual.values - fitted.values) ** 2).mean()
    rmse = np.sqrt(mse)

    log(f"[{symbol}] AR({lags}) MSE={mse:.6e}, RMSE={rmse:.6e}")
    return mse, rmse


# ------------------------
# 変化点検知
# ------------------------
def detect_change_points(df: pd.DataFrame, symbol: str, model_name="rbf", pen=10):
    prices = df["price"].values.astype(float)

    if len(prices) < 10:
        log(f"[{symbol}] データが短すぎるため変化点検知をスキップします")
        return [], [], np.array([]), pd.Series(dtype="datetime64[ns]")

    returns = np.diff(np.log(prices))
    return_times = df["timestamp"].iloc[1:].reset_index(drop=True)

    if len(returns) == 0:
        log(f"[{symbol}] return系列が空のため変化点検知をスキップします")
        return [], [], np.array([]), pd.Series(dtype="datetime64[ns]")

    signal = returns.reshape(-1, 1)

    algo = rpt.Pelt(model=model_name).fit(signal)
    bkps = algo.predict(pen=pen)

    valid_bkps = [b for b in bkps if b < len(returns)]

    log(f"[{symbol}] change points (raw): {bkps}")
    log(f"[{symbol}] change points (valid): {valid_bkps}")

    return bkps, valid_bkps, returns, return_times


# ------------------------
# 変化点候補の表示
# ------------------------
def summarize_change_points(df: pd.DataFrame, symbol: str, valid_bkps):
    print(f"\n=== {symbol} の変化点候補 ===")

    if not valid_bkps:
        print("変化点は検出されませんでした。")
        return

    for i, b in enumerate(valid_bkps, start=1):
        if b + 1 < len(df):
            t = df["timestamp"].iloc[b + 1]
            p = df["price"].iloc[b + 1]
            print(f"{i}. {t.strftime('%Y-%m-%d %H:%M:%S')} 付近, 価格={p:.6f}")


# ------------------------
# 変化点の日付抽出
# ------------------------
def extract_change_point_dates(df: pd.DataFrame, valid_bkps):
    dates = []
    for b in valid_bkps:
        if b + 1 < len(df):
            dates.append(df["timestamp"].iloc[b + 1])
    return dates


# ------------------------
# {銘柄, 日付} ペアを作成
# ------------------------
def build_symbol_date_pairs(change_point_summary: dict):
    pairs = []

    for symbol, dates in change_point_summary.items():
        for d in dates:
            pairs.append({
                "symbol": symbol,
                "date": d,
            })

    pairs.sort(key=lambda x: (x["date"], x["symbol"]))
    return pairs


# ------------------------
# {銘柄, 日付} 形式で表示
# ------------------------
def print_pair_format(change_point_summary: dict):
    print("\n==============================")
    print("変化点一覧 {銘柄, 日付}")
    print("==============================")

    pairs = build_symbol_date_pairs(change_point_summary)

    if not pairs:
        print("データなし")
        return

    for row in pairs:
        print(f"{{{row['symbol']}, {row['date'].strftime('%Y-%m-%d %H:%M:%S')}}}")


# ------------------------
# 銘柄別サマリーDataFrame作成
# ------------------------
def build_summary_dataframe(results: list):
    rows = []

    for r in results:
        cp_dates = r.get("change_points", [])
        cp_dates_str = " | ".join([d.strftime("%Y-%m-%d %H:%M:%S") for d in cp_dates]) if cp_dates else ""

        rows.append({
            "symbol": r.get("symbol"),
            "rows": r.get("rows"),
            "start_time": r.get("start_time"),
            "end_time": r.get("end_time"),
            "min_price": r.get("min_price"),
            "max_price": r.get("max_price"),
            "last_price": r.get("last_price"),
            "mse": r.get("mse"),
            "rmse": r.get("rmse"),
            "n_change_points": len(cp_dates),
            "change_points": cp_dates_str,
        })

    return pd.DataFrame(rows)


# ------------------------
# {銘柄, 日付} 用DataFrame作成
# ------------------------
def build_pairs_dataframe(change_point_summary: dict):
    pairs = build_symbol_date_pairs(change_point_summary)

    if not pairs:
        return pd.DataFrame(columns=["symbol", "date"])

    df_pairs = pd.DataFrame(pairs)
    df_pairs["date"] = df_pairs["date"].dt.strftime("%Y-%m-%d %H:%M:%S")
    return df_pairs


# ------------------------
# ランキング表示
# ------------------------
def print_rankings(summary_df: pd.DataFrame):
    print("\n==============================")
    print("ランキング")
    print("==============================")

    valid_rmse = summary_df.dropna(subset=["rmse"]).sort_values("rmse")
    if len(valid_rmse) > 0:
        print("\n[AR RMSE が小さい上位10件]")
        for i, (_, row) in enumerate(valid_rmse.head(10).iterrows(), start=1):
            print(f"{i}. {row['symbol']}  RMSE={row['rmse']:.6f}")

        print("\n[AR RMSE が大きい上位10件]")
        for i, (_, row) in enumerate(valid_rmse.sort_values("rmse", ascending=False).head(10).iterrows(), start=1):
            print(f"{i}. {row['symbol']}  RMSE={row['rmse']:.6f}")
    else:
        print("\nRMSEを計算できた銘柄がありません。")

    cp_rank = summary_df.sort_values("n_change_points", ascending=False)
    if len(cp_rank) > 0:
        print("\n[変化点が多い上位10件]")
        for i, (_, row) in enumerate(cp_rank.head(10).iterrows(), start=1):
            print(f"{i}. {row['symbol']}  変化点数={int(row['n_change_points'])}")


# ------------------------
# メイン処理
# ------------------------
def main():
    random.seed(42)
    np.random.seed(42)

    coin_files = find_coin_files(DATA_DIR)

    if NUM_COINS > len(coin_files):
        raise ValueError(f"NUM_COINS={NUM_COINS} は CSV数 {len(coin_files)} を超えています。")

    log(f"=== {DATA_DIR} からランダムに {NUM_COINS} 個選択 ===")

    selected_files = random.sample(coin_files, NUM_COINS)

    print(f"選ばれた{NUM_COINS}ファイル:")
    for f in selected_files:
        print(" -", f.name)

    dfs = {}
    results = []
    change_point_summary = {}

    # ------------------------
    # データ読み込み
    # ------------------------
    log("=== データ読み込み開始 ===")
    for idx, file_path in enumerate(selected_files, start=1):
        symbol = symbol_from_filename(file_path)

        try:
            log(f"({idx}/{NUM_COINS}) Loading {file_path.name} ...")
            df = load_coin_csv(file_path)
            dfs[symbol] = df
            log(f"[{symbol}] rows={len(df)}")
        except Exception as e:
            log(f"[{symbol}] 読み込み失敗: {e}")

    if not dfs:
        log("読み込めた銘柄がありません。終了します。")
        return

    # ------------------------
    # AR + 変化点検知
    # ------------------------
    log("=== AR・変化点検知開始 ===")
    for symbol, df in dfs.items():
        mse = None
        rmse = None
        cp_dates = []

        try:
            mse, rmse = apply_ar_price_in_sample(df, symbol, lags=AR_LAGS)
        except Exception as e:
            log(f"[{symbol}] AR失敗: {e}")

        try:
            bkps, valid_bkps, returns, return_times = detect_change_points(
                df, symbol, model_name=CHANGE_MODEL, pen=CHANGE_PEN
            )
            summarize_change_points(df, symbol, valid_bkps)
            cp_dates = extract_change_point_dates(df, valid_bkps)
            change_point_summary[symbol] = cp_dates
        except Exception as e:
            log(f"[{symbol}] 変化点検知失敗: {e}")
            change_point_summary[symbol] = []

        results.append({
            "symbol": symbol,
            "rows": len(df),
            "start_time": df["timestamp"].iloc[0].strftime("%Y-%m-%d %H:%M:%S"),
            "end_time": df["timestamp"].iloc[-1].strftime("%Y-%m-%d %H:%M:%S"),
            "min_price": float(df["price"].min()),
            "max_price": float(df["price"].max()),
            "last_price": float(df["price"].iloc[-1]),
            "mse": mse,
            "rmse": rmse,
            "change_points": cp_dates,
        })

    # ------------------------
    # 銘柄別サマリー
    # ------------------------
    summary_df = build_summary_dataframe(results)

    print("\n==============================")
    print("銘柄別サマリー")
    print("==============================")
    for _, row in summary_df.iterrows():
        print(f"\n[{row['symbol']}]")
        print(f"  rows           : {row['rows']}")
        print(f"  start_time     : {row['start_time']}")
        print(f"  end_time       : {row['end_time']}")
        print(f"  min_price      : {row['min_price']:.6f}")
        print(f"  max_price      : {row['max_price']:.6f}")
        print(f"  last_price     : {row['last_price']:.6f}")
        print(f"  mse            : {row['mse'] if pd.notna(row['mse']) else 'None'}")
        print(f"  rmse           : {row['rmse'] if pd.notna(row['rmse']) else 'None'}")
        print(f"  n_change_points: {int(row['n_change_points'])}")
        if row["change_points"]:
            print(f"  change_points  : {row['change_points']}")
        else:
            print("  change_points  : なし")

    # ------------------------
    # ランキング
    # ------------------------
    print_rankings(summary_df)

    # ------------------------
    # 最後に {銘柄, 日付} 形式で出力
    # ------------------------
    print_pair_format(change_point_summary)

    # ------------------------
    # CSV保存
    # ------------------------
    try:
        pairs_df = build_pairs_dataframe(change_point_summary)
        pairs_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        log(f"ペア形式の結果を保存しました: {OUTPUT_CSV}")
    except Exception as e:
        log(f"CSV保存失敗: {e}")


if __name__ == "__main__":
    main()

[2026-04-10 15:24:54] === d:\musashino-university\finance\coingecko_by_coin からランダムに 100 個選択 ===
選ばれた100ファイル:
 - sierra-2_sierra.csv
 - cronos-zkevm-bridged-cro-cronos-zkevm_cro.csv
 - baby-bottle_bott.csv
 - usa_usat.csv
 - gata_gata.csv
 - football-world-community_fwc.csv
 - fartful_fartful.csv
 - degate_dg.csv
 - coinxpad_cxpad.csv
 - spodermen_spoody.csv
 - proof-platform_proof.csv
 - celo-australian-dollar_audm.csv
 - sag3-ai-by-virtuals_sag3.csv
 - moth-2_moth.csv
 - based-doge_bdoge.csv
 - wrapped-microstrategy-xstock_wmstrx.csv
 - the-1-billion-horse_unicorn.csv
 - epic-chain_epic.csv
 - fight-for-justice_ffj.csv
 - paraswap_psp.csv
 - satori_sn119.csv
 - topblast_bl4st.csv
 - protoken_pro.csv
 - el-gato-3_gato.csv
 - titcoin-2_titcoin.csv
 - mooncat_mooncat.csv
 - vesper-finance_vsp.csv
 - nest-mineral-vault_nmnrl.csv
 - wifejak_wife.csv
 - acet-token_act.csv
 - diemlibre_dlb.csv
 - starknet-brother_brother.csv
 - imagen-ai_image.csv
 - radium_val.csv
 - diamond_dmd.csv
 - sola